# Data behind charts (Colab)

**[CEEW Scraping Field Course](https://mrafiuddin50.github.io/ceew_scraping_curriculum/), Module 09 (Route B) and Track B, lesson B4.**

Many sites show data only as charts, with no download button. When a chart is drawn in the browser by a charting library (Chart.js or Highcharts), the numbers are still there: the page hands them to the chart. This notebook opens the page in a hidden browser on Google's servers, waits for the charts to draw, and asks each chart for its own data. It then saves a tidy table to Google Sheets.

It does **not** work on charts that are just pictures (PNG, JPG, GIF). For those, see Module 09.

The worked example is IITM's [Air Quality Early Warning System for Delhi](https://ews.tropmet.res.in/analysis.php), whose analysis page drew 40 charts on 14 October 2026.

**Status:** the extraction code was run on the live IITM page in a browser on 14 October 2026 (40 charts read, headings matched), and the notebook was tested end to end on a test page. It has not yet been run from Colab against the live site. Step 4 reports what it found; if it finds no charts, use Route A (Claude's browser) and tell the course owner.

In [ ]:
#@title Step 1: Settings { display-mode: "form" }
PAGE_URL = "https://ews.tropmet.res.in/analysis.php" #@param {type:"string"}
#@markdown Only keep charts whose heading contains this text. Leave blank for every chart: on the IITM page that is 40 charts and about 227,000 rows.
CHART_TITLE_CONTAINS = "Hourly Forecast Verification (PM2.5)" #@param {type:"string"}
#@markdown Seconds to wait after the page loads, so the charts finish drawing.
WAIT_SECONDS = 5 #@param {type:"slider", min:2, max:30, step:1}
#@markdown Drop rows with no value (keeps the table smaller; the summary still shows them).
DROP_EMPTY_ROWS = True #@param {type:"boolean"}
SAVE_TO = "Google Sheet" #@param ["Google Sheet", "Excel file in Google Drive", "Both"]
print("Settings saved. Run Step 2.")

In [ ]:
#@title Step 2: Install a hidden browser (takes about a minute) { display-mode: "form" }
!pip -q install playwright
!playwright install --with-deps chromium > /dev/null
print("Browser installed. Run Step 3.")

In [ ]:
#@title Step 3: Load the working parts { display-mode: "form" }

import json, re, datetime as dt
import pandas as pd

IST = dt.timezone(dt.timedelta(hours=5, minutes=30))

# The JavaScript below runs inside the page, after the page has drawn its charts.
# It asks each chart for its own data: legend names, x labels and values.
EXTRACT_JS = r"""
() => {
  // Reads the numbers from every chart the page has drawn (Chart.js and Highcharts).
  // Returns one row per point: which chart, which series, the x label and the value.
  const rows = [];
  const clean = s => (s || '').replace(/\s+/g, ' ').trim();
  const headingFor = el => {
    // nearest heading above the chart in the page
    let node = el;
    while (node && node !== document.body) {
      let prev = node.previousElementSibling;
      while (prev) {
        if (/^H[1-6]$/.test(prev.tagName) && clean(prev.textContent)) return clean(prev.textContent);
        const inner = prev.querySelectorAll ? prev.querySelectorAll('h1,h2,h3,h4,h5,h6') : [];
        for (let k = inner.length - 1; k >= 0; k--) if (clean(inner[k].textContent)) return clean(inner[k].textContent);
        prev = prev.previousElementSibling;
      }
      node = node.parentElement;
    }
    return '';
  };
  const push = (lib, n, id, title, type, s, si, hidden, x, v) => {
    let value = null, text = null;
    if (typeof v === 'number') value = Number.isFinite(v) ? v : null;
    else if (typeof v === 'string' && v.trim() !== '' && !isNaN(Number(v))) value = Number(v);
    else if (v !== null && v !== undefined && v !== '') text = JSON.stringify(v);
    rows.push({ library: lib, chart_no: n, chart_id: id, chart_title: title, chart_type: type,
                series: s, series_no: si, series_hidden: hidden, x: x === undefined || x === null ? null : String(x),
                value: value, value_text: text });
  };
  // Chart.js
  if (window.Chart && typeof window.Chart.getChart === 'function') {
    let n = 0;
    document.querySelectorAll('canvas').forEach(c => {
      const ch = window.Chart.getChart(c);
      if (!ch) return;
      n++;
      const title = headingFor(c) || (ch.options && ch.options.plugins && ch.options.plugins.title && ch.options.plugins.title.text) || '';
      const labels = (ch.data && ch.data.labels) || [];
      (ch.data.datasets || []).forEach((d, si) => {
        const data = d.data || [];
        if (!data.length) push('Chart.js', n, c.id || '', title, ch.config.type, d.label || '', si, !!d.hidden, null, null);
        data.forEach((v, i) => {
          let x = labels[i];
          if (v && typeof v === 'object' && !Array.isArray(v) && ('y' in v)) { x = ('x' in v) ? v.x : x; v = v.y; }
          push('Chart.js', n, c.id || '', title, ch.config.type, d.label || '', si, !!d.hidden, x, v);
        });
      });
    });
  }
  // Highcharts
  if (window.Highcharts && Array.isArray(window.Highcharts.charts)) {
    window.Highcharts.charts.forEach((ch, k) => {
      if (!ch) return;
      const title = (ch.title && ch.title.textStr) || headingFor(ch.renderTo) || '';
      ch.series.forEach((s, si) => {
        const pts = (s.points && s.points.length) ? s.points : (s.data || []);
        if (pts.length) {
          pts.forEach(p => push('Highcharts', k + 1, ch.renderTo.id || '', title, s.type, s.name, si, !s.visible,
                                p.category !== undefined ? p.category : p.x, p.y));
        } else {
          (s.options.data || []).forEach((p, i) => {
            const x = Array.isArray(p) ? p[0] : (p && typeof p === 'object' ? p.x : i);
            const y = Array.isArray(p) ? p[1] : (p && typeof p === 'object' ? p.y : p);
            push('Highcharts', k + 1, ch.renderTo.id || '', title, s.type, s.name, si, !s.visible, x, y);
          });
        }
      });
    });
  }
  return rows;
}
"""

async def read_charts(url, wait_seconds=5):
    """Open the page in a hidden browser on Google's servers and read every chart."""
    from playwright.async_api import async_playwright
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page(user_agent="Mozilla/5.0 (compatible; CEEW-research-collector/2.0; non-commercial research)")
        response = await page.goto(url, wait_until="load", timeout=120_000)
        status = response.status if response else None
        await page.wait_for_timeout(int(wait_seconds * 1000))   # let the charts finish drawing
        rows = await page.evaluate(EXTRACT_JS)
        libs = await page.evaluate("() => ({chartjs: !!(window.Chart && window.Chart.getChart), chartjs_version: window.Chart && window.Chart.version || null, highcharts: !!(window.Highcharts && window.Highcharts.charts)})")
        canvases = await page.evaluate("() => document.querySelectorAll('canvas').length")
        await browser.close()
    return status, libs, canvases, pd.DataFrame(rows)

def tidy(df):
    """Readable x values and a unique name for each series within its chart."""
    if df.empty:
        return df
    df = df.copy()
    # Highcharts keeps dates as milliseconds since 1970 (UTC); show them as dates
    is_ms = (df["library"] == "Highcharts") & df["x"].fillna("").str.fullmatch(r"\d{12,13}")
    df.loc[is_ms, "x"] = pd.to_datetime(df.loc[is_ms, "x"].astype("int64"), unit="ms", utc=True).dt.strftime("%Y-%m-%d %H:%M UTC")
    # the same legend name can appear twice in one chart (e.g. two 'stdDev25max' bands)
    counts = df.groupby(["chart_no", "library", "series"])["series_no"].transform("nunique")
    df["series_unique"] = df["series"].where(counts == 1, df["series"] + " (#" + df["series_no"].astype(str) + ")")
    return df

def summarise(df):
    """One row per series: how many points, how many have values, first and last x."""
    if df.empty:
        return df
    g = df.groupby(["library", "chart_no", "chart_title", "series_unique"], sort=False)
    out = g.agg(points=("x", lambda s: int(s.notna().sum())),
                values=("value", lambda s: int(s.notna().sum())),
                other_values=("value_text", lambda s: int(s.notna().sum())),
                hidden=("series_hidden", "first")).reset_index()
    first_last = df[df["value"].notna()].groupby(["library", "chart_no", "series_unique"])["x"].agg(["first", "last"]).reset_index()
    out = out.merge(first_last, on=["library", "chart_no", "series_unique"], how="left")
    out = out.rename(columns={"first": "x_of_first_value", "last": "x_of_last_value"})
    out["note"] = ""
    out.loc[out["points"] == 0, "note"] = "In the legend but holds no data"
    out.loc[(out["points"] > 0) & (out["values"] == 0) & (out["other_values"] == 0), "note"] = "All values empty"
    out.loc[out["other_values"] > 0, "note"] = "Values are lists (e.g. box plot); see value_text"
    return out

def x_steps(xs):
    """Count the gaps between consecutive time labels, to spot uneven spacing."""
    t = pd.Series(pd.to_datetime(pd.Series(xs).dropna().astype(str).str.replace(" UTC", "", regex=False).unique(), errors="coerce")).dropna().reset_index(drop=True)
    if len(t) < 3:
        return "x values are not dates"
    hours = (t.diff().dropna().dt.total_seconds().abs() / 3600)
    order = "newest first" if t.iloc[0] > t.iloc[-1] else "oldest first"
    return f"{len(t)} time labels, {order}; gaps in hours: " + ", ".join(f"{h:g}h x{n}" for h, n in hours.value_counts().sort_index().items())

def wide(df_chart):
    """One row per x label, one column per series; rows oldest first."""
    df_chart = df_chart[df_chart["x"].notna()]
    w = df_chart.pivot_table(index="x", columns="series_unique", values="value", aggfunc="first", dropna=False)
    try:
        w.index = pd.to_datetime(pd.Index(w.index).astype(str).str.replace(" UTC", "", regex=False))
        w = w.sort_index()
        w.index = w.index.strftime("%Y-%m-%d %H:%M")
    except Exception:
        pass
    return w.reset_index()

print('Ready. Run Step 4.')

In [ ]:
#@title Step 4: Open the page and read its charts { display-mode: "form" }
started = dt.datetime.now(IST)
try:
    status, libs, canvases, raw = await read_charts(PAGE_URL, WAIT_SECONDS)
except Exception as e:
    status, libs, canvases, raw = None, {}, 0, pd.DataFrame()
    print(f"Could not open the page from Colab: {type(e).__name__}: {str(e)[:200]}")
    print("Use Route A in Module 09 (Claude's browser) and tell the course owner what this said.")

if status is not None:
    print(f"Page answered with status {status}. Chart libraries found: {libs}. Canvases on the page: {canvases}.")
    data = tidy(raw)
    if data.empty:
        print("No chart data found. The charts may be pictures, may need longer to draw (raise WAIT_SECONDS), or may use a library this notebook doesn't read.")
    else:
        all_titles = data.drop_duplicates(["library", "chart_no"])[["library", "chart_no", "chart_title"]]
        print(f"Charts with data: {len(all_titles)}")
        if CHART_TITLE_CONTAINS.strip():
            data = data[data["chart_title"].str.contains(CHART_TITLE_CONTAINS.strip(), case=False, regex=False)]
            print(f"Charts matching '{CHART_TITLE_CONTAINS}': {data['chart_no'].nunique()}")
            if data.empty:
                print("No chart heading matched. Headings found:")
                print(all_titles.to_string(index=False))
        print("Run Step 5.")

In [ ]:
#@title Step 5: Check what you got { display-mode: "form" }
summary = summarise(data)
print(summary[["chart_no", "chart_title", "series_unique", "points", "values", "hidden", "x_of_first_value", "x_of_last_value", "note"]].to_string(index=False))
for (lib, n), g in data.groupby(["library", "chart_no"]):
    print(f"\nChart {n} ({lib}): {g['chart_title'].iloc[0]}")
    print("  " + x_steps(g["x"]))
print("\nBefore you use these numbers:")
print("- Series marked 'In the legend but holds no data' are drawn as names only.")
print("- Hidden series are switched off on the page but still have data; decide whether you need them.")
print("- Check what time zone the times are in. The page may not say.")
print("- Spot-check three values against the chart on the live page (hover over the line).")

In [ ]:
#@title Step 6: Save to Google Sheets or Drive { display-mode: "form" }
stamp = started.strftime("%Y-%m-%d %H%M IST")
title = f"Chart data {stamp}"
long_df = data[data["value"].notna() | data["value_text"].notna()] if DROP_EMPTY_ROWS else data
about = pd.DataFrame([
    ["Page", PAGE_URL],
    ["Read (IST)", started.strftime("%Y-%m-%d %H:%M")],
    ["Charts kept", CHART_TITLE_CONTAINS or "all"],
    ["Method", "Page opened in a hidden browser (Playwright, Chromium) on Google Colab; each chart's own data read after drawing"],
    ["Rows", f"{len(long_df)} (empty rows dropped: {DROP_EMPTY_ROWS})"],
    ["Caveat", "Values as the page's charts hold them. Units and time zone come from the page, not this notebook. Verify before use in any CEEW output."],
], columns=["item", "detail"])
tabs = {"about": about, "series_summary": summary,
        "chart_data_long": long_df.drop(columns=["series"]).rename(columns={"series_unique": "series"})}
for (lib, n), g in data.groupby(["library", "chart_no"]):
    if g["value"].notna().any():
        tabs[f"{'chartjs' if lib == 'Chart.js' else 'highcharts'}{n}_wide"[:31]] = wide(g)

def to_values(df):
    clean = df.astype(object).where(pd.notna(df), "")
    return [list(map(str, df.columns))] + clean.values.tolist()

if SAVE_TO in ("Google Sheet", "Both"):
    from google.colab import auth
    auth.authenticate_user()
    import gspread
    from google.auth import default
    creds, _ = default()
    gc = gspread.authorize(creds)
    sh = gc.create(title)
    first = True
    for name, df in tabs.items():
        values = to_values(df) if len(df.columns) else [["(empty)"]]
        if first:
            ws = sh.sheet1; ws.update_title(name); first = False
        else:
            ws = sh.add_worksheet(title=name, rows=max(len(values), 1) + 5, cols=max(len(values[0]), 1) + 2)
        ws.update(values=values, range_name="A1")
    print(f"Google Sheet created: {sh.url}")

if SAVE_TO in ("Excel file in Google Drive", "Both"):
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    folder = "/content/drive/MyDrive/chart_data"
    os.makedirs(folder, exist_ok=True)
    path = f"{folder}/{title}.xlsx"
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        for name, df in tabs.items():
            df.to_excel(xw, sheet_name=name[:31], index=False)
    print(f"Saved to Google Drive: My Drive > chart_data > {title}.xlsx")

## Why not just read the page source?

On the IITM page the numbers also appear in the page source, and it's tempting to cut them out with a search. On 14 October 2026 that went wrong in three ways:

- The legend lists "WRF-Chem (400 m)", but that series holds no data. A search of the source attached the next series' numbers to it.
- The source contains a list called `model_pm25` full of numbers that the hourly PM2.5 chart does not display.
- Some chart code is commented out, so it sits in the source but never runs.

Asking the drawn chart for its data avoids all three: you get exactly what is plotted, under the name in its legend.